In [1]:
import sys
sys.path.append("..")
from modules.transforms import train_transform, val_transform, test_transform
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
from torchvision import datasets
from PIL import Image
import torch
from onnx_inference import BrainTumorClassifier
from torch.utils.data import random_split, ConcatDataset

In [10]:
session = BrainTumorClassifier("brain_tumor_classifier.onnx")
path = "../brain_tumor_dataset/Testing/pituitary_tumor/image(28).jpg"
image = Image.open(path).convert("RGB") 
output = session.predict(image)
output

Preprocessing Time: 0.902 ms
Inference Time: 3.404 ms


{'predicted_class': 'pituitary_tumor', 'confidence': 0.99999905}

In [4]:
output[0].argmax().item()

2

In [56]:
session.predict(image)

In [21]:
input_name = session.get_inputs()[0].name
path = "../brain_tumor_dataset/Testing/no_tumor/image.jpg"
image = Image.open(path).convert("RGB") 
transform = val_transform()
input_data = transform(image).unsqueeze(0).numpy()

In [22]:
outputs = session.run(None, {input_name: input_data})

In [47]:
logits = outputs[0]

In [48]:
probabilities = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
print("Probabilities:", probabilities)

Probabilities: [[5.2682927e-04 7.7970483e-04 9.9811238e-01 5.8107876e-04]]


In [44]:
predicted_class = np.argmax(probabilities, axis=1)[0]
print("Predicted class index:", predicted_class)

Predicted class index: 2


In [45]:
# Mapping to label (optional):
class_mapping = {
    0: 'glioma_tumor',
    1: 'meningioma_tumor',
    2: 'no_tumor',
    3: 'pituitary_tumor'
}

print(class_mapping[predicted_class])  # Output: meningioma_tumor

no_tumor
